# Proyecto Fin de Curso: Predicción de la Demanda Energética en España (2015-2018)

## HITO 0

**Estudiante:** Marta Requejo Merino 

**Especialidad:** Grado en IA y Big Data 

---

## 1. Definición del Problema de Machine Learning
El objetivo de este proyecto es desarrollar un sistema capaz de **predecir la demanda de energía eléctrica** en España. 

* **Tipo de Problema:** Estamos antes un problema de **Regresión**. Buscamos predecir una variable continua y numérica expresada en Megavatios (MW).
* **Utilidad Real:** En el sector eléctrico, la oferta y la demanda deben estar perfectamente balanceadas en tiempo real para evitar apagones, ya que la electricidad no se puede almacenar eficientemente a gran escala. Una predicción precisa permite a los operadores de red planificar el mix de generación óptimo, minimizando el uso de centrales térmicas de respaldo costosas y contaminantes, y maximizando la integración de energías renovables.

## 2. Identificación y Justificación de Variables

El dataset combina registros históricos de consumo energético nacional con variables climatológicas de las 5 ciudades más pobladas de España (Madrid, Barcelona, Valencia, Sevilla y Bilbao), cubriendo el periodo de 2015 a 2018.

### Variable objetivo:

- `total_load_actual`: Demanda eléctrica real en Megavatios (MW). Es la variable que nuestro modelo debe predecir.

### Variables independientes (Features):

- Temporales: `hour, month, day_of_week`. Justificadas por la naturaleza cíclica del consumo eléctrico humano (hábitos diarios, estacionalidad mensual y diferencias entre fines de semana y días laborables).

- Meteorológicas: `temp_madrid, temp_barcelona, temp_seville, temp_valencia, temp_bilbao`. Justificadas por la dependencia térmica del consumo; la temperatura es el predictor externo más potente para entender el uso de sistemas de calefacción y refrigeración.

Conforme vayamos haciendo el estudio de datos se contemplará si hay alguna variable que sea necesaria para el entrenamiento del modelo.

## 3. Arquitectura de Ingesta y Almacenamiento

Se ha diseñado e implementado una **arquitectura de procesamiento en la nube** basada en los servicios de Amazon Web Services (AWS), estructurada en diferentes capas para garantizar escalabilidad, eficiencia en costes y rendimiento analítico. La orquestación del flujo de datos y la integración de los servicios se ha programado en Python, utilizando **Boto3 (AWS SDK)** como librería principal:

1. **Capa Raw (Amazon S3):** Los archivos CSV originales se ingieren línea a línea mediante scripts locales de Python (`boto3`) y se almacenan en un bucket particionados cronológicamente (`year=YYYY/month=MM/day=DD/`). Es una capa inmutable.
2. **Crawlers:** Un Glue Crawler escanea automáticamente la capa Raw para inferir esquemas y crear tablas lógicas virtuales, evitando la configuración manual de tipos de datos.
3. **Amazon Athena:**Mediante consultas distribuidas en SQL, se realiza el cruce de datasets (JOIN), la deduplicación de energía, la limpieza de caracteres y el pivoteo matricial de las variables climáticas. Se ha optado por la potencia de este motor SQL frente a AWS Glue ETL bajo un enfoque moderno ELT (Extract, Load, Transform) porque transforma los datos al vuelo, reduciendo drásticamente los tiempos de ejecución y los costes al prescindir de clústeres dedicados.
4. **Capa Processed (Amazon S3):** Athena exporta la tabla unificada final en formato **Apache Parquet**, un almacenamiento columnar comprimido ideal para la carga masiva y veloz hacia el entorno de Jupyter Notebook en la fase de Data Science.

> **Nota:** El código fuente correspondiente a la ingesta de datos y a la orquestación de la infraestructura en la nube no se incluye en este cuaderno. Siguiendo las mejores prácticas de desarrollo, dichos procesos se han modularizado en scripts de Python independientes (archivos `.py`) que se encuentran adjuntos en el directorio `scripts/` del proyecto.


### 3.1 Justificación técnica

- **AWS** : Es el estándar de la industria en la nube. AWS ofrece escalabilidad infinita, si mañana mis datos pasan de ser 3 años a 30 años Amazon me lo permite siempre y cuando el haya suficiente presupuesto, además ofrece planes gratuitos para alumnos que permiten que proyectos como este puedan hacerse de forma gratuita.
- **S3**: Actúa como un Data Lake inmutable. Al ser el punto de entrada, garantiza que los datos raw nunca se pierdan, permitiéndo re-procesarlos tantas veces como se necesite sin dañar la fuente original.
- **Glue y Athena**: Al utilizar S3 como Data Lake inmutable, el paso lógico para el procesamiento es integrar AWS Glue y Athena, ya que ofrecen una solución serverless de alto rendimiento para transformar nuestros datasets masivos. Glue actúa como un catálogo inteligente que automatiza la detección de esquemas mediante crawlers, facilitando la gestión de cientos de columnas sin intervención manual, mientras que Athena nos permite ejecutar consultas SQL sobre grandes volúmenes de datos directamente en S3; esta combinación es ideal para limpiar, filtrar y transformar nuestros datos hacia una capa de datos procesados en formato Parquet.

- **¿Por qué no usar otro tipo de BD?** : Los datos eléctricos son masivos y crecen con el tiempo. Una base de datos relacional tradicional requeriría un servidor potente y caro. Con Athena y S3 guardamos los datos de forma más económica, teniendolos siempre disponibles, además podemos tener preparados scripts de procesamiento que se ejecutan cuando se quiera y necesite.